# โครงงานการรู้จำตัวอักษรและตัวเลขภาษาไทย 72 คลาส (Thai Character Recognition)
### วิชา Deep Learning | ตามแนวทางการสอน Practical Implementation of CNNs
---
**วัตถุประสงค์ตามเกณฑ์การให้คะแนน (15%):**
1. พัฒนาแบบจำลอง CNN ร่วมกับเทคนิคการถ่ายโอนความรู้ (**Transfer Learning: 1.5%**)
2. ใช้การสังเคราะห์ข้อมูล (**Data Augmentation: 1.5%**) ที่เหมาะสมกับภาษาไทย (ไม่กลับด้านภาพ)
3. นำเสนอแนวคิดที่น่าสนใจ (**Novel / Interesting Concept: 2.0%**): การแก้ปัญหา **Class Imbalance 72 คลาส** ด้วย Class-Weighted Loss และ Cosine Annealing LR Scheduler
4. ประสิทธิภาพการทำนายภาพทดสอบ (**Test Accuracy: 5%**) และการจัดลำดับคะแนน (**Ranking: 3%**)


## 1. การเตรียมสภาพแวดล้อมและโมดูลที่จำเป็น (Environment & Imports)


In [ ]:
import os
import sys
import json
import time
import math
import random
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

# ตั้งค่า Random Seed เพื่อให้ผลลัพธ์สามารถทำซ้ำได้ (Reproducibility)
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


## 2. การเตรียมชุดข้อมูลและการวิเคราะห์ Class Imbalance (72 คลาส)


In [ ]:
# โหลดข้อมูล metadata ที่ผ่านการแบ่ง Stratified Split (80% Train, 20% Val)
metadata_file = 'dataset_metadata.csv'
if not os.path.exists(metadata_file):
    print(f"Running dataset preparation...")
    os.system('python prepare_dataset.py')

df = pd.read_csv(metadata_file)
print(f"Total dataset samples: {len(df)}")
print(f"Training samples: {len(df[df['split'] == 'train'])}")
print(f"Validation samples: {len(df[df['split'] == 'val'])}")
print(f"Total Classes: {df['class_code'].nunique()}")

# โหลด mapping ตัวอักษร
with open('char_mapping.json', 'r', encoding='utf-8') as f:
    char_map = json.load(f)

classes = sorted(df['class_code'].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for i, c in enumerate(classes)}

# แสดงตัวอย่างการกระจายข้อมูล คลาสที่มีมากสุด vs น้อยสุด
class_counts = df['class_code'].value_counts()
print(f"คลาสที่มีตัวอย่างมากที่สุด: {class_counts.index[0]} ({char_map.get(str(class_counts.index[0]), {}).get('char', '')}) = {class_counts.iloc[0]} ภาพ")
print(f"คลาสที่มีตัวอย่างน้อยที่สุด: {class_counts.index[-1]} ({char_map.get(str(class_counts.index[-1]), {}).get('char', '')}) = {class_counts.iloc[-1]} ภาพ")


## 3. สร้าง Custom Dataset 
โครงสร้างคลาส `ThaiCharacterDataset` มี 3 เมธอดหลักคือ `__init__`, `__len__`, และ `__getitem__`


In [ ]:
class ThaiCharacterDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        label = class_to_idx[row['class_code']]

        # โหลดภาพเป็น RGB
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label


## 4. การสังเคราะห์ข้อมูล (Data Augmentation)
> ** ระวังสำคัญสำหรับตัวอักษรไทย:** **ห้ามใช้ RandomHorizontalFlip และ RandomVerticalFlip เด็ดขาด** 
> เพราะจะทำให้ความหมายของตัวอักษรเปลี่ยนไป เช่น ด กลายเป็น ค หรือ ภ กลายเป็น ถ 
> ดังนั้นเราจึงใช้ **RandomRotation (±10°)**, **RandomAffine (เลื่อน/เอียงลายมือ)** และ **ColorJitter (ความเข้มของหมึก)**


In [ ]:
img_size = 224

train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.06, 0.06), shear=6),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# สร้าง Dataset และ DataLoader
train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']

train_dataset = ThaiCharacterDataset(train_df, transform=train_transform)
val_dataset = ThaiCharacterDataset(val_df, transform=val_transform)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


## 5. กำหนดแบบจำลอง Convolutional Neural Network (Transfer Learning)
เลือกใช้สถาปัตยกรรมหลัก **ResNet-50 (Residual Network)** สำหรับจำแนก 72 คลาส
โดยนำเอา Pretrained weights มาและเปลี่ยน Classifier Head สำหรับ 72 คลาส


In [ ]:
# แบบจำลองหลัก: ResNet-50 (Transfer Learning จาก ImageNet)
def create_model(model_name="resnet50", num_classes=72):
    if model_name == "resnet50":
        model = resnet50(weights=ResNet50_Weights.DEFAULT)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )
    return model

model = create_model("resnet50", num_classes=len(classes))
model = model.to(device)
print(f"Model initialized: {model.__class__.__name__} for {len(classes)} classes.")


## 6. กำหนด Loss Function และ Optimizer
- **Class-Weighted Cross-Entropy Loss:** คำนวณน้ำหนัก $w_c = \frac{1}{\sqrt{N_c}}$ เพื่อชดเชย Class Imbalance
- **AdamW Optimizer** ร่วมกับ **Cosine Annealing Learning Rate Scheduler**


In [ ]:
# คำนวณ Class Weights เพื่อแก้ปัญหา Class Imbalance
train_counts = train_df['class_code'].value_counts()
class_sample_counts = [train_counts.get(c, 1) for c in classes]
weights = [1.0 / math.sqrt(max(count, 1)) for count in class_sample_counts]
weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)
weights_tensor = weights_tensor / weights_tensor.sum() * len(classes)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

epochs = 3
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
print("Configured Class-Weighted Cross-Entropy Loss and Cosine Annealing Optimizer.")


## 7. ขั้นตอนการฝึกสอนแบบจำลอง (Training Execution)


In [ ]:
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_top1_acc': [], 'val_top3_acc': []}

for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    scheduler.step()
    train_loss = running_loss / total
    train_acc = (correct / total) * 100.0
    
    # Validation Loop
    model.eval()
    val_loss, val_top1_correct, val_top3_correct, val_total = 0.0, 0, 0, 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, top1_preds = torch.max(outputs, 1)
            val_top1_correct += (top1_preds == labels).sum().item()
            
            _, top3_preds = outputs.topk(3, 1, True, True)
            val_top3_correct += sum([labels[i] in top3_preds[i] for i in range(labels.size(0))])
            val_total += labels.size(0)
            
    val_loss = val_loss / val_total
    val_top1_acc = (val_top1_correct / val_total) * 100.0
    val_top3_acc = (val_top3_correct / val_total) * 100.0
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_top1_acc'].append(val_top1_acc)
    history['val_top3_acc'].append(val_top3_acc)
    
    print(f"Epoch [{epoch}/{epochs}] | Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Top-1: {val_top1_acc:.2f}%, Top-3: {val_top3_acc:.2f}%")
    
    if val_top1_acc > best_val_acc:
        best_val_acc = val_top1_acc
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"--> Saved best model with Val Top-1: {best_val_acc:.2f}%")


###  Save the model that we have trained

บันทึกค่าน้ำหนัก (Model Weights) ของแบบจำลองที่ผ่านการฝึกสอนแล้วลงในไฟล์ `model.pt` และ `best_model.pt` เพื่อนำไปใช้งานในขั้นตอน Inference


In [ ]:
# Save model that we have trained
torch.save(model.state_dict(), 'model.pt')
torch.save(model.state_dict(), 'best_model.pt')
print("Successfully saved model weights to 'model.pt' and 'best_model.pt'!")


## 8. กราฟแสดงประสิทธิภาพแบบจำลอง (Training Curves)


In [ ]:
plt.figure(figsize=(12, 4.5))

plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), history['train_loss'], label='Train Loss', color='blue', marker='o')
plt.plot(range(1, epochs + 1), history['val_loss'], label='Val Loss', color='red', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curves')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), history['train_acc'], label='Train Acc', color='blue', marker='o')
plt.plot(range(1, epochs + 1), history['val_top1_acc'], label='Val Top-1 Acc', color='green', marker='s')
plt.plot(range(1, epochs + 1), history['val_top3_acc'], label='Val Top-3 Acc', color='orange', linestyle='--', marker='^')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title(f'Accuracy Curves (Best Top-1: {best_val_acc:.2f}%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300)
plt.show()
